### 1) Required installs

In [1]:
!pip -q install transformers accelerate sentencepiece pandas torch

### 2) Imports

In [2]:
import os
import re
import json
import shutil
from pathlib import Path

import pandas as pd
import torch

from transformers import pipeline

PROJECT_DIR = Path("/content/maintainers-copilot")
SPLITS_DIR = PROJECT_DIR / "data" / "splits"
REPORTS_DIR = PROJECT_DIR / "data" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cpu


### 3) Upload test split

Uploads the local data/splits/test.jsonl so we can test NER and summarization on real issue text.

In [3]:
from google.colab import files

print("Upload test.jsonl from your local data/splits folder.")
uploaded = files.upload()

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

for filename in uploaded.keys():
    target = SPLITS_DIR / filename
    shutil.move(filename, target)
    print("Saved:", target)

Upload test.jsonl from your local data/splits folder.


Saving test.jsonl to test.jsonl
Saved: /content/maintainers-copilot/data/splits/test.jsonl


### 4) Load test dataset as a dataframe

In [4]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

test_df = load_jsonl(SPLITS_DIR / "test.jsonl")

print("Rows:", len(test_df))
print("Columns:", test_df.columns.tolist())
print(test_df[["title", "mapped_label"]].head())

Rows: 100
Columns: ['external_id', 'repo', 'number', 'title', 'body', 'state', 'labels', 'mapped_label', 'created_at', 'closed_at', 'url']
                                             title mapped_label
0                          Unknown flag --replicas     question
1                     Building local cluster error     question
2  Watch API always returns Waiting for Pod Status          bug
3                                  pd.sh is flaky.          bug
4             Kubernetes configuration file schema     question


### 5) Build issue text

Combines title and body into one text field.

In [5]:
def clean_text(value):
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value)

def build_issue_text(row):
    title = clean_text(row.get("title", ""))
    body = clean_text(row.get("body", ""))
    return f"Title: {title}\n\nBody:\n{body}"

test_df["text"] = test_df.apply(build_issue_text, axis=1)

sample_text = test_df.iloc[0]["text"]
print(sample_text[:2000])

Title: Unknown flag --replicas

Body:
This fails

```
⧑ kubectl resize --replicas 2 rc cloud-api-stream                                                                                                                                             
unknown flag: --replicas
```

This works

```
⧑ kubectl resize --replicas=2 rc cloud-api-stream                                                                                                                                             
resized
```

Both should work, no?



### 6) Define rule-based code-shaped NER

Extracts useful maintainer entities like file paths, versions, commands, errors, URLs, functions, and Kubernetes-style components. This satisfies the “code-shaped entities” idea.

In [6]:
ENTITY_PATTERNS = {
    "url": r"https?://[^\s\)\]\}]+",
    "file_path": r"(?:[A-Za-z0-9_\-\.]+/)+[A-Za-z0-9_\-\.]+\.[A-Za-z0-9]+",
    "python_file": r"\b[A-Za-z_][A-Za-z0-9_]*\.py\b",
    "go_file": r"\b[A-Za-z_][A-Za-z0-9_]*\.go\b",
    "yaml_file": r"\b[A-Za-z_][A-Za-z0-9_\-]*\.ya?ml\b",
    "json_file": r"\b[A-Za-z_][A-Za-z0-9_\-]*\.json\b",
    "version": r"\bv?\d+\.\d+(?:\.\d+)?(?:[-+][A-Za-z0-9\.\-]+)?\b",
    "error_type": r"\b[A-Za-z_][A-Za-z0-9_]*(?:Error|Exception|Failure|Timeout)\b",
    "command": r"(?:^|\n)\s*(?:kubectl|docker|python|pip|npm|go|make|helm|git)\s+[^\n]+",
    "function_call": r"\b[A-Za-z_][A-Za-z0-9_]*\([^\)]{0,80}\)",
    "class_or_type": r"\b[A-Z][A-Za-z0-9_]{2,}\b",
    "k8s_component": r"\b(?:kubelet|kubectl|apiserver|scheduler|controller-manager|etcd|pod|deployment|service|ingress|configmap|secret|namespace)\b",
    "issue_ref": r"#\d+",
    "sha": r"\b[0-9a-f]{7,40}\b",
}

def extract_entities(text: str, max_per_type: int = 20):
    results = []

    for entity_type, pattern in ENTITY_PATTERNS.items():
        matches = re.finditer(pattern, text, flags=re.IGNORECASE | re.MULTILINE)

        seen = set()
        count = 0

        for match in matches:
            value = match.group(0).strip()

            if not value:
                continue

            normalized = value.lower()

            if normalized in seen:
                continue

            seen.add(normalized)

            results.append({
                "type": entity_type,
                "value": value,
                "start": match.start(),
                "end": match.end(),
            })

            count += 1
            if count >= max_per_type:
                break

    results = sorted(results, key=lambda x: (x["start"], x["type"]))
    return results

entities = extract_entities(sample_text)

print("Entities found:", len(entities))
for e in entities[:40]:
    print(e)

Entities found: 18
{'type': 'class_or_type', 'value': 'Title', 'start': 0, 'end': 5}
{'type': 'class_or_type', 'value': 'Unknown', 'start': 7, 'end': 14}
{'type': 'class_or_type', 'value': 'flag', 'start': 15, 'end': 19}
{'type': 'class_or_type', 'value': 'replicas', 'start': 22, 'end': 30}
{'type': 'class_or_type', 'value': 'Body', 'start': 32, 'end': 36}
{'type': 'class_or_type', 'value': 'This', 'start': 38, 'end': 42}
{'type': 'class_or_type', 'value': 'fails', 'start': 43, 'end': 48}
{'type': 'class_or_type', 'value': 'kubectl', 'start': 56, 'end': 63}
{'type': 'k8s_component', 'value': 'kubectl', 'start': 56, 'end': 63}
{'type': 'class_or_type', 'value': 'resize', 'start': 64, 'end': 70}
{'type': 'class_or_type', 'value': 'cloud', 'start': 87, 'end': 92}
{'type': 'class_or_type', 'value': 'api', 'start': 93, 'end': 96}
{'type': 'class_or_type', 'value': 'stream', 'start': 97, 'end': 103}
{'type': 'class_or_type', 'value': 'works', 'start': 280, 'end': 285}
{'type': 'class_or_type

### 7) Test NER on several issues

Checks that the extractor works across multiple real issues.

In [7]:
ner_examples = []

for idx, row in test_df.head(10).iterrows():
    text = row["text"]
    entities = extract_entities(text)

    ner_examples.append({
        "id": row.get("id"),
        "title": row.get("title"),
        "label": row.get("mapped_label"),
        "entities": entities,
    })

for ex in ner_examples[:3]:
    print("=" * 100)
    print("Title:", ex["title"])
    print("Label:", ex["label"])
    print("Entities:")
    for entity in ex["entities"][:20]:
        print(" ", entity["type"], "=>", entity["value"])

Title: Unknown flag --replicas
Label: question
Entities:
  class_or_type => Title
  class_or_type => Unknown
  class_or_type => flag
  class_or_type => replicas
  class_or_type => Body
  class_or_type => This
  class_or_type => fails
  class_or_type => kubectl
  k8s_component => kubectl
  class_or_type => resize
  class_or_type => cloud
  class_or_type => api
  class_or_type => stream
  class_or_type => works
  class_or_type => resized
  class_or_type => Both
  class_or_type => should
  class_or_type => work
Title: Building local cluster error
Label: question
Entities:
  class_or_type => Title
  class_or_type => Building
  class_or_type => local
  class_or_type => cluster
  class_or_type => error
  class_or_type => Body
  class_or_type => since
  class_or_type => clone
  class_or_type => the
  class_or_type => source
  class_or_type => ubuntu
  class_or_type => server
  version => 14.04
  class_or_type => with
  class_or_type => docker
  class_or_type => etcd
  k8s_component => etcd
  

### 8) Save NER check report

Saves NER examples for your Day 2 evidence/reporting.

In [8]:
ner_report_path = REPORTS_DIR / "ner_check_examples.jsonl"

with ner_report_path.open("w", encoding="utf-8") as f:
    for row in ner_examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", ner_report_path)

Saved: /content/maintainers-copilot/data/reports/ner_check_examples.jsonl


### 9) Load summarization model

Loads a small pretrained summarization model. This is not fine-tuned; it is used as an integration tool.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SUMMARIZER_MODEL = "sshleifer/distilbart-cnn-12-6"

summarizer_tokenizer = AutoTokenizer.from_pretrained(SUMMARIZER_MODEL)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZER_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
summarizer_model = summarizer_model.to(device)
summarizer_model.eval()

print("Loaded summarizer:", SUMMARIZER_MODEL)
print("Device:", device)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loaded summarizer: sshleifer/distilbart-cnn-12-6
Device: cpu


### 10) Define summarization helper

In [11]:
def summarize_issue_text(text: str, max_chars: int = 3500):
    text = clean_text(text)

    if len(text.strip()) == 0:
        return ""

    clipped = text[:max_chars]

    inputs = summarizer_tokenizer(
        clipped,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    ).to(device)

    with torch.no_grad():
        output_ids = summarizer_model.generate(
            **inputs,
            max_new_tokens=130,
            min_new_tokens=30,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )

    summary = summarizer_tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return summary.strip()

summary = summarize_issue_text(sample_text)

print("Original:")
print(sample_text[:1500])

print("\nSummary:")
print(summary)

Original:
Title: Unknown flag --replicas

Body:
This fails

```
⧑ kubectl resize --replicas 2 rc cloud-api-stream                                                                                                                                             
unknown flag: --replicas
```

This works

```
⧑ kubectl resize --replicas=2 rc cloud-api-stream                                                                                                                                             
resized
```

Both should work, no?


Summary:
Title: Unknown flag --replicas.iopiopiopel: --Replicas.2rc cloud-api-stream. The Unknown flag: --replica.iopel. The Unidentified flag:.iopel. The Unknown Flag.


### 11) Run summarizer on sample issues

Tests summarization on several issues from different labels.

In [12]:
summary_examples = []

sample_df = (
    test_df
    .groupby("mapped_label", group_keys=False)
    .head(2)
    .reset_index(drop=True)
)

for idx, row in sample_df.iterrows():
    text = row["text"]

    try:
        summary = summarize_issue_text(text)
        error = None
    except Exception as e:
        summary = ""
        error = str(e)

    summary_examples.append({
        "id": row.get("id"),
        "title": row.get("title"),
        "label": row.get("mapped_label"),
        "summary": summary,
        "error": error,
    })

for ex in summary_examples:
    print("=" * 100)
    print("Title:", ex["title"])
    print("Label:", ex["label"])
    print("Summary:", ex["summary"])
    if ex["error"]:
        print("Error:", ex["error"])

Title: Unknown flag --replicas
Label: question
Summary: Title: Unknown flag --replicas.iopiopiopel: --Replicas.2rc cloud-api-stream. The Unknown flag: --replica.iopel. The Unidentified flag:.iopel. The Unknown Flag.
Title: Building local cluster error
Label: question
Summary: Building local cluster error: Building go targets for linux/amd64. Error in /root/kubernetes/hack/lib/golang.sh:303: 'go install "${goflags[@]:+$goflag[@]]." -ldflags "${version_ldflags}" "${binaries[@]""; "go build github.com/GoogleCloudPlatform/kubecfg: signal: killed"
Title: Watch API always returns Waiting for Pod Status
Label: bug
Summary: Watch API always returns 'Waiting' for Pod status, but REST API provides a different value which appears to be accurate (Pending, Running, etc. etc.)
Title: pd.sh is flaky.
Label: bug
Summary: 62 tests passed, and of the failing 39 tests we had 20 failures for `pd.sh. I suspect most of the other failures are related to etcd pressure (but this is just speculation right now a

### 12) Save summarizer check report

Saves summarizer outputs for later comparison and documentation.

In [13]:
summary_report_path = REPORTS_DIR / "summarizer_check_examples.jsonl"

with summary_report_path.open("w", encoding="utf-8") as f:
    for row in summary_examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", summary_report_path)

Saved: /content/maintainers-copilot/data/reports/summarizer_check_examples.jsonl


### 13) Combined tool-style function

Simulates the future model-server behavior: one input issue text gives NER entities and a summary.

In [14]:
def analyze_issue_for_tools(title: str, body: str):
    text = f"Title: {clean_text(title)}\n\nBody:\n{clean_text(body)}"

    entities = extract_entities(text)

    try:
        summary = summarize_issue_text(text)
    except Exception as e:
        summary = ""
        return {
            "summary": summary,
            "entities": entities,
            "error": str(e),
        }

    return {
        "summary": summary,
        "entities": entities,
        "error": None,
    }

example_row = test_df.iloc[0]

tool_output = analyze_issue_for_tools(
    title=example_row.get("title", ""),
    body=example_row.get("body", ""),
)

print(json.dumps(tool_output, indent=2, ensure_ascii=False)[:4000])

{
  "summary": "Title: Unknown flag --replicas.iopiopiopel: --Replicas.2rc cloud-api-stream. The Unknown flag: --replica.iopel. The Unidentified flag:.iopel. The Unknown Flag.",
  "entities": [
    {
      "type": "class_or_type",
      "value": "Title",
      "start": 0,
      "end": 5
    },
    {
      "type": "class_or_type",
      "value": "Unknown",
      "start": 7,
      "end": 14
    },
    {
      "type": "class_or_type",
      "value": "flag",
      "start": 15,
      "end": 19
    },
    {
      "type": "class_or_type",
      "value": "replicas",
      "start": 22,
      "end": 30
    },
    {
      "type": "class_or_type",
      "value": "Body",
      "start": 32,
      "end": 36
    },
    {
      "type": "class_or_type",
      "value": "This",
      "start": 38,
      "end": 42
    },
    {
      "type": "class_or_type",
      "value": "fails",
      "start": 43,
      "end": 48
    },
    {
      "type": "class_or_type",
      "value": "kubectl",
      "start": 56,
    

### 14) Save final combined examples

Creates one final report showing both tool outputs together.

In [15]:
combined_examples = []

for idx, row in test_df.head(10).iterrows():
    output = analyze_issue_for_tools(
        title=row.get("title", ""),
        body=row.get("body", ""),
    )

    combined_examples.append({
        "id": row.get("id"),
        "title": row.get("title"),
        "label": row.get("mapped_label"),
        "tool_output": output,
    })

combined_report_path = REPORTS_DIR / "ner_summarizer_combined_examples.jsonl"

with combined_report_path.open("w", encoding="utf-8") as f:
    for row in combined_examples:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", combined_report_path)

Saved: /content/maintainers-copilot/data/reports/ner_summarizer_combined_examples.jsonl


### 15) Create implementation notes

Creates a small markdown note explaining the design choices. This is useful later for DECISIONS.md.

In [16]:
notes = f"""# Day 2 NER and Summarizer Check

## Purpose

This notebook validates the behavior of two NLP tools required by the Maintainer's Copilot:

- NER-style extraction of code-shaped entities from issue text
- Summarization of issue title and body

These are integration tools, not separately fine-tuned models.

## NER approach

The NER tool uses deterministic regular expressions to extract code-shaped entities.

Entity types include:

- URLs
- file paths
- Python, Go, YAML, and JSON files
- versions
- error and exception names
- shell commands
- function calls
- class/type-looking names
- Kubernetes components
- issue references
- commit SHAs

This approach is simple, explainable, fast, and appropriate for code-shaped issue text.

## Summarizer approach

The summarizer uses a pretrained model:

{SUMMARIZER_MODEL}

The model is used as an integration tool to produce concise maintainer-facing issue summaries.

## Limitations

- Rule-based NER may miss unusual entity formats.
- Rule-based NER may over-extract capitalized words as class/type names.
- The summarizer is pretrained and not fine-tuned on Kubernetes issues.
- Long issue bodies are clipped before summarization.

## Files produced

- data/reports/ner_check_examples.jsonl
- data/reports/summarizer_check_examples.jsonl
- data/reports/ner_summarizer_combined_examples.jsonl
"""

notes_path = REPORTS_DIR / "ner_summarizer_notes.md"

with notes_path.open("w", encoding="utf-8") as f:
    f.write(notes)

print("Saved:", notes_path)
print(notes)

Saved: /content/maintainers-copilot/data/reports/ner_summarizer_notes.md
# Day 2 NER and Summarizer Check

## Purpose

This notebook validates the behavior of two NLP tools required by the Maintainer's Copilot:

- NER-style extraction of code-shaped entities from issue text
- Summarization of issue title and body

These are integration tools, not separately fine-tuned models.

## NER approach

The NER tool uses deterministic regular expressions to extract code-shaped entities.

Entity types include:

- URLs
- file paths
- Python, Go, YAML, and JSON files
- versions
- error and exception names
- shell commands
- function calls
- class/type-looking names
- Kubernetes components
- issue references
- commit SHAs

This approach is simple, explainable, fast, and appropriate for code-shaped issue text.

## Summarizer approach

The summarizer uses a pretrained model:

sshleifer/distilbart-cnn-12-6

The model is used as an integration tool to produce concise maintainer-facing issue summaries.



### 16) Zip outputs

Creates a zip file containing only the reports from this notebook.

In [17]:
ARTIFACT_ZIP = "/content/day2_ner_summarizer_check_artifacts.zip"

if os.path.exists(ARTIFACT_ZIP):
    os.remove(ARTIFACT_ZIP)

shutil.make_archive(
    base_name=ARTIFACT_ZIP.replace(".zip", ""),
    format="zip",
    root_dir=PROJECT_DIR,
    base_dir="data/reports",
)

print("Created:", ARTIFACT_ZIP)

Created: /content/day2_ner_summarizer_check_artifacts.zip


In [18]:
from google.colab import files

files.download(ARTIFACT_ZIP)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>